# Benchmarking Data Science Agents
## Comparative Analysis Pipeline

**Tools:** PyArrow · DuckDB · LangGraph · Docker  
**Data:** 35,000+ rows across 4 benchmarks — Chatbot Arena, MLE-Bench, SWE-bench, GAIA

This notebook walks through the full pipeline:
1. Data Collection
2. PyArrow: CSV → Parquet conversion
3. DuckDB: SQL analytics
4. Analysis: clustering, win rates, capability gaps
5. Visualizations
6. Results & Interpretation

## Step 1: Setup & Imports

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print('PyArrow version:', pa.__version__)
print('DuckDB version:', duckdb.__version__)
print('Setup complete!')

## Step 2: Data Collection

We scraped 4 benchmarks using HuggingFace Datasets API and Kaggle API.
Raw data is stored as CSVs in `data/raw/`.

In [ ]:
raw_dir = Path('data/raw')

# Show what raw files we have
for f in raw_dir.glob('*.csv'):
    df = pd.read_csv(f)
    print(f'{f.name}: {len(df):,} rows, {len(df.columns)} columns')

In [ ]:
# Preview each dataset
for f in raw_dir.glob('*.csv'):
    print(f'\n--- {f.stem.upper()} ---')
    df = pd.read_csv(f)
    print(df.head(3))

## Step 3: PyArrow — CSV to Parquet Conversion

**Why Parquet?**
- Columnar format: DuckDB only reads columns it needs
- Snappy compression: 3-5x smaller than CSV
- Schema embedded: no re-inferring types on every read
- Native to both PyArrow and DuckDB: zero-copy sharing

In [ ]:
import pyarrow.csv as pa_csv

processed_dir = Path('data/processed')

# Show Parquet files already created by collect.py
parquet_files = list(processed_dir.glob('bench_*.parquet'))
print(f'Parquet files in data/processed/:')
for f in parquet_files:
    table = pq.read_table(f)
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}: {table.num_rows:,} rows, {size_kb:.1f} KB')

In [ ]:
# Demonstrate PyArrow zero-copy: read CSV vs Parquet speed comparison
import time

csv_path = raw_dir / 'chatbot_arena.csv'
parquet_path = processed_dir / 'bench_chatbot_arena.parquet'

# CSV read time
start = time.time()
df_csv = pd.read_csv(csv_path)
csv_time = time.time() - start

# Parquet read time  
start = time.time()
table = pq.read_table(parquet_path)
parquet_time = time.time() - start

print(f'CSV read time:     {csv_time:.3f}s')
print(f'Parquet read time: {parquet_time:.3f}s')
print(f'Speedup: {csv_time/parquet_time:.1f}x faster')

## Step 4: DuckDB — SQL Analytics on Parquet

**Why DuckDB?**
- In-process OLAP engine — no server needed
- Reads Parquet natively (zero memory copies)
- Full SQL: window functions, PIVOT, CTEs, aggregations
- Replaces all pandas groupby/merge operations

In [ ]:
conn = duckdb.connect(':memory:')
conn.execute('INSTALL parquet; LOAD parquet;')

# Query directly on Parquet files — no loading into memory
stats = conn.execute(f"""
    SELECT
        benchmark,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT agent) AS unique_agents,
        ROUND(AVG(score), 4) AS mean_score,
        ROUND(STDDEV(score), 4) AS std_score
    FROM read_parquet('{processed_dir}/unified_data.parquet')
    GROUP BY benchmark
    ORDER BY total_rows DESC
""").df()

print('Benchmark Statistics (computed by DuckDB SQL):')
print(stats.to_string(index=False))

In [ ]:
# DuckDB window functions — top agents per benchmark
top_agents = conn.execute(f"""
    WITH ranked AS (
        SELECT
            benchmark,
            agent,
            ROUND(AVG(score), 4) AS mean_score,
            COUNT(*) AS n_tasks,
            DENSE_RANK() OVER (
                PARTITION BY benchmark
                ORDER BY AVG(score) DESC
            ) AS rank
        FROM read_parquet('{processed_dir}/unified_data.parquet')
        WHERE agent IS NOT NULL
        GROUP BY benchmark, agent
    )
    SELECT * FROM ranked WHERE rank <= 5
    ORDER BY benchmark, rank
""").df()

print('Top 5 Agents per Benchmark:')
print(top_agents.to_string(index=False))

In [ ]:
# Chatbot Arena win rate analysis using DuckDB UNION ALL
winrates = conn.execute(f"""
    WITH model_scores AS (
        SELECT model_a AS model,
            CASE WHEN winner = 'model_a' THEN 1.0
                 WHEN winner = 'tie' THEN 0.5
                 ELSE 0.0 END AS won
        FROM read_parquet('{processed_dir}/bench_chatbot_arena.parquet')
        WHERE model_a IS NOT NULL
        UNION ALL
        SELECT model_b AS model,
            CASE WHEN winner = 'model_b' THEN 1.0
                 WHEN winner = 'tie' THEN 0.5
                 ELSE 0.0 END AS won
        FROM read_parquet('{processed_dir}/bench_chatbot_arena.parquet')
        WHERE model_b IS NOT NULL
    )
    SELECT
        model,
        COUNT(*) AS total_battles,
        ROUND(AVG(won) * 100, 2) AS win_rate_pct
    FROM model_scores
    GROUP BY model
    HAVING COUNT(*) >= 10
    ORDER BY win_rate_pct DESC
    LIMIT 10
""").df()

print('Top 10 Models by Win Rate (Chatbot Arena):')
print(winrates.to_string(index=False))

## Step 5: Analysis Results

Clustering, capability gaps and statistics computed by `analyze.py`

In [ ]:
# Load analysis results
benchmark_stats = pd.read_parquet(processed_dir / 'benchmark_stats.parquet')
cluster_labels  = pd.read_parquet(processed_dir / 'cluster_labels.parquet')
gaps            = pd.read_parquet(processed_dir / 'capability_gaps.parquet')

print('Benchmark Stats:')
print(benchmark_stats[['benchmark','total_rows','unique_agents','mean_score']].to_string(index=False))
print(f'\nClusters: {cluster_labels["kmeans_cluster"].nunique()} clusters, {len(cluster_labels)} agents')
print(f'Gaps: {len(gaps)} rows')

In [ ]:
import json

# Show silhouette scores
with open('outputs/tables/silhouette_scores.json') as f:
    sil = json.load(f)

print('Silhouette Scores by K:')
for k, v in sil.items():
    print(f'  K={k}: {v}')
print(f'\nBest K: {max(sil, key=sil.get)} (score={max(sil.values())})')

## Step 6: Visualizations

In [ ]:
from IPython.display import Image, display
from pathlib import Path

figures = list(Path('outputs/figures').glob('*.png'))
print(f'{len(figures)} figures generated:')
for f in figures:
    print(f'  {f.name}')

In [ ]:
# Show benchmark sizes
display(Image('outputs/figures/benchmark_sizes.png'))

In [ ]:
# Show Chatbot Arena win rates
display(Image('outputs/figures/chatbot_arena_winrates.png'))

In [ ]:
# Show MLE-bench score distributions
display(Image('outputs/figures/mle_bench_scores.png'))

In [ ]:
# Show agent clusters
display(Image('outputs/figures/agent_clusters.png'))

In [ ]:
# Show GAIA difficulty distribution
display(Image('outputs/figures/gaia_difficulty.png'))

In [ ]:
# Show dendrogram
display(Image('outputs/figures/dendrogram.png'))

## Step 7: Key Findings & Interpretation

### What the benchmarks reveal:

1. **Chatbot Arena** — Measures human preference. Very subjective. 20 models across 33k battles. GPT-4 class models dominate win rates.

2. **MLE-bench** — Most rigorous benchmark. Real Kaggle competitions with objective scoring. Score variance is enormous (std=25,508) showing how domain-specific ML performance is.

3. **SWE-bench** — Task complexity varies hugely by GitHub repo (143 to 24,770 chars). Shows that not all code repair tasks are equal.

4. **GAIA** — Most structured benchmark. Clean 3-level difficulty progression. Smallest dataset (165 tasks) but highest quality design.

### Clustering insight:
- 5 distinct agent clusters with silhouette score of 0.95+ 
- Very high silhouette means agents clearly separate by capability profile
- Benchmarks are NOT interchangeable — each captures a different dimension

### Big Data Stack contribution:
- PyArrow: 3-5x faster than pandas for CSV reading
- DuckDB: SQL on Parquet without loading into memory
- LangGraph: Reproducible 6-stage pipeline with error handling
- Docker: Fully containerised, runs anywhere

## Step 8: Run Full Pipeline via LangGraph Agent

In [ ]:
import subprocess
result = subprocess.run(['python', 'agent.py', '--from', 'analyze'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)